# 04 XGBoost

Gradient-boosted trees to capture combinations of signals that logistic regression can't.

**Setup**
- Number of trees chosen by early stopping on November–December 2019 (211,481 transactions), after fitting on January–October 2019 (713,369). Validation stays untouched.
- Final model retrained on all of 2019 with that tree count.
- Compared no class weighting against `scale_pos_weight`.

**Findings**
- Unweighted: 676 trees, PR-AUC 0.976. Weighted: 992 trees, PR-AUC 0.974.
- Precision at 80% / 90% recall: 0.993 / 0.959 unweighted, vs 0.990 / 0.950 weighted.
- The unweighted model's average score (0.0061) matches the validation fraud rate (0.0061), so its probabilities are calibrated on average.
- Weighting was slower and no better, so the unweighted model is saved as `models/xgb_v1.joblib`.

In [1]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")
import numpy as np, pandas as pd
from src.data import load_train_val
from src.features import NUMERIC, CATEGORICAL, FEATURES

tr, val = load_train_val("../data/raw/fraudTrain.csv")

inner_cut = pd.Timestamp("2019-11-01")
fit_part  = tr[tr["trans_date_trans_time"] <  inner_cut]
stop_part = tr[tr["trans_date_trans_time"] >= inner_cut]
print(f"fit: {len(fit_part):,} | early-stop: {len(stop_part):,} | val: {len(val):,}")

fit: 713,369 | early-stop: 211,481 | val: 371,825


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score, precision_recall_curve
from xgboost import XGBClassifier

def make_prep():
    return ColumnTransformer([
        ("num", "passthrough", NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
    ])

BASE_PARAMS = dict(learning_rate=0.05, max_depth=6, subsample=0.8,
                   colsample_bytree=0.8, eval_metric="aucpr",
                   n_jobs=-1, random_state=42)

def train_xgb(scale_pos_weight):
    # Step 1: find how many trees to use, stopping when the early-stop slice stops improving
    prep = make_prep().fit(fit_part[FEATURES])
    finder = XGBClassifier(n_estimators=2000, early_stopping_rounds=100,
                           scale_pos_weight=scale_pos_weight, **BASE_PARAMS)
    finder.fit(prep.transform(fit_part[FEATURES]), fit_part["is_fraud"],
               eval_set=[(prep.transform(stop_part[FEATURES]), stop_part["is_fraud"])],
               verbose=False)
    n_trees = finder.best_iteration + 1

    # Step 2: retrain on all of 2019 with that many trees
    model = Pipeline([
        ("prep", make_prep()),
        ("clf", XGBClassifier(n_estimators=n_trees,
                              scale_pos_weight=scale_pos_weight, **BASE_PARAMS)),
    ])
    model.fit(tr[FEATURES], tr["is_fraud"])
    return model, n_trees

In [3]:
neg, pos = (tr["is_fraud"] == 0).sum(), (tr["is_fraud"] == 1).sum()
results = {}
for name, spw in [("unweighted", 1.0), ("weighted", neg / pos)]:
    model, n_trees = train_xgb(spw)
    scores = model.predict_proba(val[FEATURES])[:, 1]
    results[name] = (model, scores)
    print(f"{name:10s} | trees {n_trees:4d} | PR-AUC {average_precision_score(val['is_fraud'], scores):.3f} "
          f"| avg score {scores.mean():.4f} (val fraud rate {val['is_fraud'].mean():.4f})")

def precision_at_recall(y, scores, target):
    precision, recall, _ = precision_recall_curve(y, scores)
    return precision[recall >= target].max()

for name, (_, scores) in results.items():
    row = " | ".join(f"recall {r:.0%}: precision {precision_at_recall(val['is_fraud'], scores, r):.3f}"
                     for r in [0.7, 0.8, 0.9])
    print(f"{name:10s} | {row}")

unweighted | trees  676 | PR-AUC 0.976 | avg score 0.0061 (val fraud rate 0.0061)
weighted   | trees  992 | PR-AUC 0.974 | avg score 0.0082 (val fraud rate 0.0061)
unweighted | recall 70%: precision 0.998 | recall 80%: precision 0.993 | recall 90%: precision 0.959
weighted   | recall 70%: precision 0.997 | recall 80%: precision 0.990 | recall 90%: precision 0.950


In [4]:
import joblib
best = max(results, key=lambda k: average_precision_score(val["is_fraud"], results[k][1]))
joblib.dump(results[best][0], "../models/xgb_v1.joblib")
print("Saved:", best)

Saved: unweighted
